# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
import os
from glob import glob

# Write your code below.
import pandas as pd
import random


# Load the PRICE_DATA environment variable
PRICE_DATA = os.getenv("PRICE_DATA")

# Use glob to find all parquet files recursively within PRICE_DATA
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)

print(f"Found {len(parquet_files)} parquet files in {PRICE_DATA}")



Found 2711 parquet files in ../../05_src/data/prices/


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.
# Read all parquet files into a Dask DataFrame; set ticker as the index
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

# For each ticker group: sort by Date, compute Close_lag_1 and Adj_Close_lag_1
dd_feat = (
    dd_px
    .groupby("ticker", group_keys=False)
    .apply(
        lambda x: x.sort_values("Date", ascending=True).assign(
            Close_lag_1     = x["Close"].shift(1),
            Adj_Close_lag_1 = x["Adj Close"].shift(1)
        ),
        meta=pd.DataFrame(
            data={
                "Date":             "datetime64[ns]",
                "Open":             "f8",
                "High":             "f8",
                "Low":              "f8",
                "Close":            "f8",
                "Adj Close":        "f8",
                "Volume":           "i8",
                "source":           "object",
                "Year":             "int32",
                "Close_lag_1":      "f8",
                "Adj_Close_lag_1":  "f8",
            },
            index=pd.Index([], dtype=pd.StringDtype(), name="ticker")
        )
    )
)

# Add returns and hi_lo_range using vectorised Dask assign
dd_feat = dd_feat.assign(
    returns     = lambda x: (x["Close"] / x["Close_lag_1"]) - 1,
    hi_lo_range = lambda x: x["High"] - x["Low"]
)

dd_feat


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=89,,,,,,,,,,,,,
AC,object,object,object,object,object,object,object,object,object,object,object,object,object
ACHV,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZOM,...,...,...,...,...,...,...,...,...,...,...,...,...
ZOM,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [6]:
# Write your code below.

# Trigger Dask computation and materialise into a Pandas DataFrame
df_feat = dd_feat.compute()

# Sort by ticker then Date so the rolling window respects time order
df_feat = df_feat.sort_values(["ticker", "Date"]).reset_index()

# Compute the 10-day rolling mean of returns, grouped per ticker
# min_periods=1 avoids NaN for the first few rows of each ticker
df_feat["returns_ma_10"] = (
    df_feat
    .groupby("ticker")["returns"]
    .transform(lambda x: x.rolling(10, min_periods=1).mean())
)

print(df_feat.shape)
df_feat[["ticker", "Date", "Close", "returns", "returns_ma_10"]].head(15)

(311737, 15)


,ticker,Date,Close,returns,returns_ma_10
0,AC,2007-12-27,53.869999,NaN,NaN
1,AC,2007-12-28,280.010010,4.197884,4.197884
2,AC,2007-12-31,280.010010,0.000000,2.098942
3,AC,2008-01-02,270.097992,-0.035399,1.387495
4,AC,2008-01-03,252.751999,-0.064221,1.024566
5,AC,2008-01-04,52.799999,-0.791100,0.661433
6,AC,2008-01-07,234.167999,3.435000,1.123694
7,AC,2008-01-08,234.167999,0.000000,0.963166
8,AC,2008-01-09,237.884995,0.015873,0.844755
9,AC,2008-01-10,244.078995,0.026038,0.753786


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

### Answer

**Was it necessary to convert to Pandas?**  
No — it was not strictly necessary. Dask does support `.rolling()` and `.groupby().transform()` operations, so the moving average could have been computed entirely within Dask before calling `.compute()`.

**Would it have been better to do it in Dask?**  
It depends on the scale of the data:

- **For a small dataset** (as in this assignment — a sample of ~60 tickers), converting to Pandas first is perfectly fine and actually simpler. Pandas' rolling window functions are well-optimised and the full dataset easily fits in memory.

- **For a large dataset** (thousands of tickers, many years of daily data), it would be better to compute the rolling mean *within* Dask before calling `.compute()`. This keeps the heavy computation distributed and avoids loading the entire dataset into memory at once. The key Dask constraint to be aware of is that a rolling window applied across partition boundaries can silently lose rows at each boundary; the correct approach is to use `map_partitions` with an appropriate overlap (`dask.dataframe.rolling`) or to ensure each ticker's data lives in a single partition.

In summary: for small data, convert to Pandas and use `.rolling()` there — it is simpler and less error-prone. For big data, stay in Dask and use `dd.groupby().transform()` with a rolling window to avoid memory constraints.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.